# Data Cleaning and Transformation

Prepares the autoimmune dataset for a **multiclass diagnosis model** (115 diseases + `Normal`).

Order of operations (it matters, to avoid data leakage):

1. Load and inspect
2. Clean (IDs, missing values, column names, constant columns)
3. Encode categorical features and targets
4. **Split into train / test first**
5. **Fit the scaler on the training set only**, then apply it to both sets
6. Validate and save the outputs for the modelling notebook

In [1]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 1. Load and inspect

In [2]:
FILE_NAME = "Autoimmune_Disorder_10k_with_All_Disorders.csv"

# Works from the notebook folder (Exploratory/), the project root, or next to the CSV
candidates = [
    os.path.join("..", "data", FILE_NAME),
    os.path.join("data", FILE_NAME),
    FILE_NAME,
]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Could not find {FILE_NAME} in: {candidates}")

OUT_DIR = os.path.join(os.path.dirname(DATA_PATH) or ".", "processed")
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (12500, 79)


,Patient_ID,Age,Gender,Diagnosis,Sickness_Duration_Months,RBC_Count,Hemoglobin,Hematocrit,MCV,MCH,MCHC,RDW,Reticulocyte_Count,WBC_Count,Neutrophils,Lymphocytes,Monocytes,Eosinophils,Basophils,PLT_Count,MPV,ANA,Esbach,MBL_Level,ESR,C3,C4,CRP,Anti-dsDNA,Anti-Sm,Rheumatoid factor,ACPA,Anti-TPO,Anti-Tg,Anti-SMA,Low-grade fever,Fatigue or chronic tiredness,Dizziness,Weight loss,Rashes and skin lesions,Stiffness in the joints,Brittle hair or hair loss,Dry eyes and/or mouth,General 'unwell' feeling,Joint pain,Anti_dsDNA,Anti_enterocyte_antibodies,anti_LKM1,Anti_RNP,ASCA,Anti_Ro_SSA,Anti_CBir1,Anti_BP230,Anti_tTG,DGP,Anti_BP180,ASMA,Anti_IF,IgG_IgE_receptor,Anti_SRP,Anti_desmoglein_3,Anti_La_SSB,Anti_Jo1,ANCA,anti_centromere,Anti_desmoglein_1,EMA,Anti_type_VII_collagen,C1_inhibitor,Anti_TIF1,Anti_epidermal_basement_membrane_IgA,Anti_OmpC,pANCA,Anti_tissue_transglutaminase,anti_Scl_70,Anti_Mi2,Anti_parietal_cell,Progesterone_antibodies,Anti_Sm
0,1,65,Female,Linear IgA disease,26,3.54,13.18,44.35,99.85,31.80,33.12,12.52,2.08,8278,72.25,19.80,9.26,4.02,1.08,429117,8.08,1,1.50,1.81,40,1.32,0.49,3.10,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0,1,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
1,2,61,Male,Dermatomyositis,106,4.97,14.73,42.39,94.86,29.94,31.07,15.15,1.21,8592,56.73,42.68,2.66,1.67,1.16,487184,8.92,1,1.76,1.08,23,0.40,0.17,2.73,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1,0,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
2,3,42,Male,Ord's thyroiditis,6,3.91,13.66,42.41,88.26,29.43,35.61,15.52,2.68,8719,50.92,31.21,6.96,4.32,1.07,499962,7.70,1,2.49,1.42,44,0.95,0.26,1.02,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,45,Female,Restless legs syndrome,11,4.57,14.96,40.10,85.95,27.42,33.93,12.14,1.79,7698,41.20,33.60,6.54,3.12,0.72,129793,10.95,1,0.75,1.05,25,1.06,0.26,5.45,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1,0,1,1,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,5,78,Female,Autoimmune polyendocrine syndrome type 2 (APS2),30,4.88,11.21,37.58,93.16,29.87,31.29,15.65,1.42,9755,45.82,42.57,3.05,4.60,1.40,467390,10.51,1,0.76,1.14,42,0.73,0.25,1.05,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0,1,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("Patient_ID unique:", df["Patient_ID"].is_unique)

nulls = df.isnull().sum()
print("\nColumns with missing values:")
print(nulls[nulls > 0])

Duplicate rows: 0
Patient_ID unique: True

Columns with missing values:
Anti-dsDNA           2500
Anti-Sm              2500
Rheumatoid factor    2500
ACPA                 2500
Anti-TPO             2500
Anti-Tg              2500
Anti-SMA             2500
dtype: int64


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Patient_ID,12500.0,6250.500000,3608.583517,1.00,3125.7500,6250.50,9375.2500,12500.0
Age,12500.0,49.757040,17.740705,18.00,34.0000,50.00,65.0000,84.0
Sickness_Duration_Months,12500.0,47.964960,39.034342,0.00,8.0000,45.00,82.0000,119.0
RBC_Count,12500.0,4.449722,0.572939,3.50,3.9800,4.46,4.8300,5.9
Hemoglobin,12500.0,13.124339,1.842016,10.00,11.6100,13.14,14.4100,17.5
Hematocrit,12500.0,41.559422,3.318242,36.00,38.8800,41.54,43.8100,50.0
MCV,12500.0,90.175951,8.150511,75.01,83.3875,90.30,97.0600,105.0
MCH,12500.0,29.021782,1.628080,26.00,27.6900,29.03,30.3825,32.0
MCHC,12500.0,33.605753,1.413444,31.00,32.4300,33.62,34.8300,36.0
RDW,12500.0,13.788202,1.175959,11.50,12.8000,13.73,14.7200,16.0


In [5]:
df.dtypes.value_counts()

int64      51
float64    26
str         2
Name: count, dtype: int64

## 2. Cleaning

### 2.1 Missing antibody values
The 7 antibody-panel columns are missing for exactly the 2,500 `Normal` rows (the panel was never run on controls).
Before imputing, verify that this is really the only pattern.

In [6]:
PANEL_COLS_RAW = [
    "Anti-dsDNA", "Anti-Sm", "Rheumatoid factor", "ACPA",
    "Anti-TPO", "Anti-Tg", "Anti-SMA",
]

is_normal = df["Diagnosis"].eq("Normal")
assert df.loc[is_normal, PANEL_COLS_RAW].isna().all().all(), "Some Normal rows have panel results"
assert df.loc[~is_normal, PANEL_COLS_RAW].notna().all().all(), "Some disease rows are missing panel results"
print(f"Panel is missing for all {is_normal.sum()} Normal rows and for no disease rows.")

Panel is missing for all 2500 Normal rows and for no disease rows.


### 2.2 Clean the data
* Drop `Patient_ID` (identifier, not a feature).
* The raw file has **two different columns with near-identical names**: the 7-test lab panel (`Anti-dsDNA`, `Anti-Sm`)
  and the disease-specific flags (`Anti_dsDNA`, `Anti_Sm`). Standardising names to snake_case would make them collide,
  so the panel columns get a `Panel_` prefix.
* Names are also made safe for XGBoost / LightGBM, which reject characters such as `'`, `/` and spaces.
* Panel NaNs → 0 ("not tested / negative"), cast to int.
* Drop zero-variance columns.

In [7]:
df_clean = df.copy()

# Identifier
assert df_clean["Patient_ID"].is_unique
df_clean = df_clean.drop(columns=["Patient_ID"])

# Column names
def clean_name(col):
    return re.sub(r"[^0-9a-zA-Z]+", "_", col).strip("_")

rename_map = {c: clean_name(f"Panel_{c}" if c in PANEL_COLS_RAW else c) for c in df_clean.columns}
df_clean = df_clean.rename(columns=rename_map)
assert df_clean.columns.is_unique, "Column names collide after cleaning"
PANEL_COLS = [rename_map[c] for c in PANEL_COLS_RAW]

# Missing antibody-panel values (checked in 2.1)
df_clean[PANEL_COLS] = df_clean[PANEL_COLS].fillna(0).astype(int)

# Zero-variance columns carry no information
constant_cols = [c for c in df_clean.columns if df_clean[c].nunique(dropna=False) == 1]
print("Dropping constant columns:", constant_cols)
df_clean = df_clean.drop(columns=constant_cols)

# Sanity checks
assert df_clean.isnull().sum().sum() == 0
assert (df_clean.select_dtypes("number") >= 0).all().all(), "Unexpected negative values"
print("Shape after cleaning:", df_clean.shape)
df_clean[PANEL_COLS].isnull().sum()

Dropping constant columns: ['Anti_tTG']
Shape after cleaning: (12500, 77)


Panel_Anti_dsDNA           0
Panel_Anti_Sm              0
Panel_Rheumatoid_factor    0
Panel_ACPA                 0
Panel_Anti_TPO             0
Panel_Anti_Tg              0
Panel_Anti_SMA             0
dtype: int64

## 3. Encode categorical features and targets
* `Gender` → `Gender_Encoded` (Female = 0, Male = 1)
* `Diagnosis` → `Diagnosis_Encoded` (0–115, multiclass target) and `Is_Autoimmune` (binary target)

In [8]:
df_clean["Gender_Encoded"] = df_clean["Gender"].map({"Female": 0, "Male": 1})
assert df_clean["Gender_Encoded"].notna().all(), "Unexpected Gender value"
df_clean["Gender_Encoded"] = df_clean["Gender_Encoded"].astype(int)

# Binary target: 0 = Normal, 1 = Autoimmune disorder
df_clean["Is_Autoimmune"] = (df_clean["Diagnosis"] != "Normal").astype(int)

# Multiclass target
label_encoder = LabelEncoder()
df_clean["Diagnosis_Encoded"] = label_encoder.fit_transform(df_clean["Diagnosis"])
print("Number of classes:", len(label_encoder.classes_))

label_map = pd.DataFrame({
    "Diagnosis_Encoded": range(len(label_encoder.classes_)),
    "Diagnosis": label_encoder.classes_,
})
label_map.head()

Number of classes: 116


,Diagnosis_Encoded,Diagnosis
0,0,Acute disseminated encephalomyelitis
1,1,Acute motor axonal neuropathy
2,2,Addison's disease
3,3,Alopecia areata
4,4,Anti-NMDA receptor encephalitis


## 4. Separate features from targets and split **before** scaling
Targets are kept out of `X` so they cannot leak into the model. The split is stratified on the diagnosis because
the classes are imbalanced (`Normal` ≈ 20 %, every disease ≈ 0.5–1.5 %).

In [9]:
TARGET_COLS = ["Diagnosis", "Diagnosis_Encoded", "Is_Autoimmune"]

CONTINUOUS_COLS = [
    "Age", "Sickness_Duration_Months",
    "RBC_Count", "Hemoglobin", "Hematocrit", "MCV", "MCH", "MCHC", "RDW", "Reticulocyte_Count",
    "WBC_Count", "Neutrophils", "Lymphocytes", "Monocytes", "Eosinophils", "Basophils",
    "PLT_Count", "MPV", "Esbach", "MBL_Level", "ESR", "C3", "C4", "CRP",
]

X = df_clean.drop(columns=TARGET_COLS).drop(columns=["Gender"])
y = df_clean[["Diagnosis_Encoded", "Is_Autoimmune"]]

# Everything that is not continuous must be a 0/1 flag
BINARY_COLS = [c for c in X.columns if c not in CONTINUOUS_COLS]
assert all(X[c].isin([0, 1]).all() for c in BINARY_COLS), "Non-binary column outside CONTINUOUS_COLS"
assert set(CONTINUOUS_COLS) <= set(X.columns)

X = X.astype({c: "float64" for c in CONTINUOUS_COLS})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y["Diagnosis_Encoded"],
    random_state=RANDOM_STATE,
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Features: {len(CONTINUOUS_COLS)} continuous + {len(BINARY_COLS)} binary = {X.shape[1]}")

Train: (10000, 76)  Test: (2500, 76)
Features: 24 continuous + 52 binary = 76


## 5. Standardise continuous features
The scaler is **fit on the training set only** and then applied to the test set, so no information from the test set
leaks into training. Binary flags are left as 0/1, and scaled values **replace** the originals (no duplicate
`*_scaled` columns, which would double every feature and create perfect multicollinearity).

Tree-based models (Random Forest, XGBoost, ...) are unaffected by scaling; it matters for logistic regression, SVM, kNN and neural nets.

In [10]:
scaler = StandardScaler()
scaler.fit(X_train[CONTINUOUS_COLS])

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[CONTINUOUS_COLS] = scaler.transform(X_train[CONTINUOUS_COLS])
X_test_scaled[CONTINUOUS_COLS] = scaler.transform(X_test[CONTINUOUS_COLS])

print("Transformation complete!")
print(f"Original shape: {df.shape} -> X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")
X_train_scaled.head()

Transformation complete!
Original shape: (12500, 79) -> X_train: (10000, 76), X_test: (2500, 76)


,Age,Sickness_Duration_Months,RBC_Count,Hemoglobin,Hematocrit,MCV,MCH,MCHC,RDW,Reticulocyte_Count,WBC_Count,Neutrophils,Lymphocytes,Monocytes,Eosinophils,Basophils,PLT_Count,MPV,ANA,Esbach,MBL_Level,ESR,C3,C4,CRP,Panel_Anti_dsDNA,Panel_Anti_Sm,Panel_Rheumatoid_factor,Panel_ACPA,Panel_Anti_TPO,Panel_Anti_Tg,Panel_Anti_SMA,Low_grade_fever,Fatigue_or_chronic_tiredness,Dizziness,Weight_loss,Rashes_and_skin_lesions,Stiffness_in_the_joints,Brittle_hair_or_hair_loss,Dry_eyes_and_or_mouth,General_unwell_feeling,Joint_pain,Anti_dsDNA,Anti_enterocyte_antibodies,anti_LKM1,Anti_RNP,ASCA,Anti_Ro_SSA,Anti_CBir1,Anti_BP230,DGP,Anti_BP180,ASMA,Anti_IF,IgG_IgE_receptor,Anti_SRP,Anti_desmoglein_3,Anti_La_SSB,Anti_Jo1,ANCA,anti_centromere,Anti_desmoglein_1,EMA,Anti_type_VII_collagen,C1_inhibitor,Anti_TIF1,Anti_epidermal_basement_membrane_IgA,Anti_OmpC,pANCA,Anti_tissue_transglutaminase,anti_Scl_70,Anti_Mi2,Anti_parietal_cell,Progesterone_antibodies,Anti_Sm,Gender_Encoded
11127,1.362308,-1.229378,1.262706,1.743092,0.955857,-0.881720,-0.436965,0.674308,-1.647957,-0.540998,0.877850,1.351909,1.049330,-0.352169,0.886272,-0.906646,1.167550,-0.207105,0,-1.456869,-0.499924,-0.837920,0.378042,0.320113,-1.204087,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10153,0.630571,-1.229378,0.197165,2.040865,1.076423,0.914364,-0.197975,-0.139326,-0.964786,-1.207822,-1.195597,-0.732981,0.582032,-1.361101,0.778428,-0.906646,-0.213591,-0.695994,0,-1.508565,-1.771930,-0.763219,0.621276,-0.204933,-1.278865,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
6562,-0.889189,0.540511,-1.043054,-0.649920,-0.882774,-0.262592,-0.093800,0.773359,-1.041643,0.404962,-0.003567,-0.580349,-1.632129,0.463944,0.418951,-0.730358,0.504573,-1.724852,1,-1.133773,0.083948,0.880217,-1.880564,0.695145,0.554840,1,1,0,1,0,1,0,1,1,1,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3592,-1.283201,0.514861,-0.169660,0.432890,-0.882774,1.745590,-1.386796,-0.174701,-1.383228,1.599043,-0.814760,0.929735,-0.654717,0.925811,0.886272,-1.188708,1.533605,0.770675,1,0.430011,-0.791860,0.581410,-1.324599,-1.480044,0.984006,0,0,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6384,1.080871,1.489582,0.756137,0.367922,-1.542873,-0.535989,0.224853,-0.146401,1.417773,1.180339,-0.216044,0.945160,1.581462,-1.316260,0.823363,1.173562,-0.380930,-1.294337,1,0.106915,-0.896122,0.058499,-1.498338,-1.105011,0.314248,0,1,1,1,1,1,1,0,1,1,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


## 6. Validation

In [11]:
# No missing values, everything numeric
assert X_train_scaled.isnull().sum().sum() == 0 and X_test_scaled.isnull().sum().sum() == 0
assert not X_train_scaled.select_dtypes(exclude="number").columns.tolist()

# Scaler was fit on train only: train ~ mean 0 / std 1, test is close but not identical
print("Train mean (max abs):", X_train_scaled[CONTINUOUS_COLS].mean().abs().max().round(6))
print("Train std  (min/max):", X_train_scaled[CONTINUOUS_COLS].std().min().round(3),
      X_train_scaled[CONTINUOUS_COLS].std().max().round(3))
print("Test mean  (max abs):", X_test_scaled[CONTINUOUS_COLS].mean().abs().max().round(3))

# Targets are not in the features
assert not set(TARGET_COLS) & set(X_train_scaled.columns)

# Stratification worked: every class is in both sets, with matching proportions
tr = y_train["Diagnosis_Encoded"].value_counts(normalize=True).sort_index()
te = y_test["Diagnosis_Encoded"].value_counts(normalize=True).sort_index()
assert len(tr) == len(te) == len(label_encoder.classes_)
print("Max class-proportion difference train vs test:", (tr - te).abs().max().round(4))
print("Normal share  train/test:", y_train["Is_Autoimmune"].eq(0).mean().round(3), y_test["Is_Autoimmune"].eq(0).mean().round(3))

Train mean (max abs): 0.0
Train std  (min/max): 1.0 1.0
Test mean  (max abs): 0.065
Max class-proportion difference train vs test: 0.0003
Normal share  train/test: 0.2 0.2


## 7. Save outputs for the modelling notebook

In [12]:
train_out = pd.concat([X_train_scaled, y_train], axis=1)
test_out = pd.concat([X_test_scaled, y_test], axis=1)

train_out.to_csv(os.path.join(OUT_DIR, "train_transformed.csv"), index=False)
test_out.to_csv(os.path.join(OUT_DIR, "test_transformed.csv"), index=False)
label_map.to_csv(os.path.join(OUT_DIR, "label_mapping.csv"), index=False)

# Fitted objects, so new patients can be transformed identically at prediction time
joblib.dump(scaler, os.path.join(OUT_DIR, "scaler.joblib"))
joblib.dump(label_encoder, os.path.join(OUT_DIR, "label_encoder.joblib"))
with open(os.path.join(OUT_DIR, "feature_config.json"), "w") as f:
    json.dump({"feature_order": X_train_scaled.columns.tolist(),
               "continuous_cols": CONTINUOUS_COLS,
               "binary_cols": BINARY_COLS}, f, indent=2)

print("Saved to:", os.path.abspath(OUT_DIR))
print(sorted(os.listdir(OUT_DIR)))

Saved to: /Users/snehatanwar/Documents/Final year Project/Implementation/autoimmune_project/data/processed
['feature_config.json', 'label_encoder.joblib', 'label_mapping.csv', 'scaler.joblib', 'test_transformed.csv', 'train_transformed.csv']


## Notes for the modelling step
* **Targets:** `Diagnosis_Encoded` (116 classes) or `Is_Autoimmune` (binary). Use only one as `y`, and never leave the other in `X`.
* **Class imbalance:** `Normal` is 20 % of rows, each disease only ~0.5–1.5 %. Use `class_weight="balanced"` (or sample weights),
  stratified cross-validation on the training set, and report macro-F1 / per-class recall, not just accuracy. Resample (e.g. SMOTE) only *inside* CV folds, never before the split.
* **`Normal` is trivially separable in this dataset.** All control rows have `Sickness_Duration_Months = 0`, `ANA = 0`, no symptoms,
  and a never-run antibody panel. Overall accuracy will therefore look better than it is; evaluate disease-vs-disease performance separately.
* **Limited signal among diseases:** ~41 % of disease rows have no disease-specific antibody flag, and the lab values look near-random across
  diagnoses, so expect modest multiclass accuracy.
* **`Gender` is not consistent with some diagnoses** (e.g. males with endometriosis or oophoritis), which suggests synthetic data; treat results accordingly.
* **New data:** transform it with the saved `scaler.joblib`, using the column order in `feature_config.json`.